<a href="https://colab.research.google.com/github/Zelenenene/git-and-github-final-project-Zelenenene/blob/main/MSE641_Task2_T5_first_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets sentencepiece evaluate rouge_score
!pip install rouge-score

import pandas as pd
import numpy as np
import torch
import random
import evaluate

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)

from transformers import Seq2SeqTrainingArguments

from transformers import Seq2SeqTrainer

from tqdm import tqdm

from transformers import T5Tokenizer, T5ForConditionalGeneration

import matplotlib.pyplot as plt

from transformers import set_seed

from nltk.translate.bleu_score import corpus_bleu

from rouge_score import rouge_scorer

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

#Read the training set first
train = pd.read_json("train.jsonl", lines=True)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.3 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=67778de05776c232e138dcb5011f40b64c512ba2f000c18061569965419c8992
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
#See the first few rows for the training set

train.head()

,uuid,postId,postText,postPlatform,targetParagraphs,targetTitle,targetDescription,targetKeywords,targetMedia,targetUrl,provenance,spoiler,spoilerPositions,tags
0,0af11f6b-c889-4520-9372-66ba25cb7657,532quh,"[Wes Welker Wanted Dinner With Tom Brady, But ...",reddit,[It’ll be just like old times this weekend for...,"Wes Welker Wanted Dinner With Tom Brady, But P...",It'll be just like old times this weekend for ...,"new england patriots, ricky doyle, top stories,","[http://pixel.wp.com/b.gif?v=noscript, http://...",http://nesn.com/2016/09/wes-welker-wanted-dinn...,"{'source': 'anonymized', 'humanSpoiler': 'They...",[how about that morning we go throw?],"[[[3, 151], [3, 186]]]",[passage]
1,b1a1f63d-8853-4a11-89e8-6b2952a393ec,411701128456593408,[NASA sets date for full recovery of ozone hole],Twitter,[2070 is shaping up to be a great year for Mot...,Hole In Ozone Layer Expected To Make Full Reco...,2070 is shaping up to be a great year for Moth...,"ozone layer,ozone hole determined by weather,M...",[http://s.m.huffpost.com/assets/Logo_Huffingto...,http://huff.to/1cH672Z,"{'source': 'anonymized', 'humanSpoiler': '2070...",[2070],"[[[0, 0], [0, 4]]]",[phrase]
2,008b7b19-0445-4e16-8f9e-075b73f80ca4,380537005123190784,[This is what makes employees happy -- and it'...,Twitter,"[Despite common belief, money isn't the key to...",Intellectual Stimulation Trumps Money For Empl...,By: Chad Brooks \r\nPublished: 09/18/2013 06:4...,"employee happiness money,employee happiness in...",[http://i.huffpost.com/gen/1359674/images/o-HA...,http://huff.to/1epfeaw,"{'source': 'anonymized', 'humanSpoiler': 'Inte...",[intellectual stimulation],"[[[1, 186], [1, 210]]]",[phrase]
3,31ecf93c-3e21-4c80-949b-aa549a046b93,844567852531286016,[Passion is overrated — 7 work habits you need...,Twitter,"[It’s common wisdom. Near gospel really, and n...","‘Follow your passion’ is wrong, here are 7 hab...",There's a lot more to work that loving your job,"business, work-life, careers",None,None,"{'source': 'anonymized', 'humanSpoiler': None,...",[Purpose connects us to something bigger and i...,"[[[11, 25], [11, 101]], [[17, 56], [17, 85]], ...",[multi]
4,31b108a3-c828-421a-a4b9-cf651e9ac859,814186311573766144,[The perfect way to cook rice so that it's per...,Twitter,"[Boiling rice may seem simple, but there is a ...",Revealed: The perfect way to cook rice so that...,The question 'How does one cook rice properly?...,"Quora,users,share,perfect,way,cook,rice",None,None,"{'source': 'anonymized', 'humanSpoiler': None,...",[in a rice cooker],"[[[5, 60], [5, 76]]]",[phrase]


In [ ]:
#Also, see the column names of the train dataset
train.columns

Index(['uuid', 'postId', 'postText', 'postPlatform', 'targetParagraphs',
       'targetTitle', 'targetDescription', 'targetKeywords', 'targetMedia',
       'targetUrl', 'provenance', 'spoiler', 'spoilerPositions', 'tags'],
      dtype='object')

Based on the dataset description, 14 variables are available. The target variable is spoiler

In [ ]:
#Read the test dataset as well as the validation set
test = pd.read_json("test.jsonl", lines=True)

val = pd.read_json("val.jsonl", lines=True)

#See what variables are available in the test dataset
test.columns

Index(['postId', 'postText', 'postPlatform', 'targetParagraphs', 'targetTitle',
       'targetDescription', 'targetKeywords', 'targetMedia', 'targetUrl',
       'id'],
      dtype='object')

For the input variables, we mainly focus on postText, targetTitle, and targetParagraphs. Then we need to change the targetParagraphs from list type to str type.

Then we can do data analysis first

In [ ]:
train["targetParagraphs"].apply(len).describe()

,targetParagraphs
count,3200.000000
mean,14.195625
std,14.154305
min,1.000000
25%,6.000000
50%,10.000000
75%,17.000000
max,249.000000


This means that most spoilers appear in the first half of the article, especially within the first 25%. We can choose the length to be 6 in this case.

In [ ]:
#Here, according to the histogram above, we consider to limit the number of paragraphs we use.
#In this case, if the paragraphs length is smaller than 6, we can use all of them.
def combine_paragraphs_first6(paragraphs):
    if len(paragraphs) <= 6:
        return "\n\n".join(paragraphs)

    return "\n\n".join(paragraphs[:6])


train["context"] = train["targetParagraphs"].apply(combine_paragraphs_first6)

val["context"] = val["targetParagraphs"].apply(combine_paragraphs_first6)

test["context"] = test["targetParagraphs"].apply(combine_paragraphs_first6)

Similarly, we need to do the same for postText

In [ ]:
def combine_text(x):
    if isinstance(x, list):
        return " ".join(x)
    return x

train["postText"] = train["postText"].apply(combine_text)

val["postText"] = val["postText"].apply(combine_text)

test["postText"] = test["postText"].apply(combine_text)

Then we need to transform the format of the spoiler

In [ ]:
def process_spoiler(spoiler):
    return " ".join(spoiler)

train["target_text"] = train["spoiler"].apply(process_spoiler)

val["target_text"] = val["spoiler"].apply(process_spoiler)

So the type of spoiler changes. Next, we create the T5 input

In [ ]:
def create_input(row):

    return (
        "generate spoiler: "
        + row["postText"]
        + " title: "
        + row["targetTitle"]
        + " context: "
        + row["context"]
    )


train["input_text"] = train.apply(create_input, axis=1)

val["input_text"] = val.apply(create_input, axis=1)

test["input_text"] = test.apply(create_input, axis=1)

Next, before the formal tokenizer is applied, we need to check and confirm how many pieces of text exceed the maximum length allowed by T5.

In [ ]:
#Load the tokenizater first
tokenizer = T5Tokenizer.from_pretrained("t5-base")

#check the length
train_lengths = train["input_text"].apply(
    lambda x: len(tokenizer(x)["input_ids"])
)

print(train_lengths.describe())

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

count    3200.000000
mean      347.846250
std       152.039691
min        27.000000
25%       248.750000
50%       325.000000
75%       415.000000
max      1975.000000
Name: input_text, dtype: float64


In [ ]:
(train_lengths > 1024).sum()

np.int64(16)

In [ ]:
#We need to check the percentage that exceed max length
(train_lengths > 1024).mean()

np.float64(0.005)

This means that 0.5% of the training samples exceed the maximum input length of 1024 tokens that the model can handle.

Then, we can create a HuggingFace Dataset.

In [ ]:
max_input_length = 1024

train_dataset = Dataset.from_pandas(
    train[["input_text", "target_text"]]
)

val_dataset = Dataset.from_pandas(
    val[["input_text", "target_text"]]
)

test_dataset = Dataset.from_pandas(
    test[["input_text"]]
)

#The generated spoiler is limited to a maximum of 64 tokens.
max_target_length = 64

target_lengths = train["target_text"].apply(
    lambda x: len(tokenizer(x)["input_ids"])
)

target_lengths.describe()

,target_text
count,3200.000000
mean,21.924375
std,26.988551
min,2.000000
25%,5.000000
50%,12.000000
75%,30.000000
max,337.000000


In [ ]:
(target_lengths > 64).mean()

np.float64(0.0603125)

Only about 5% of the spoilers exceed the token length of 64, so set it to 64 is reasonable.

Then, we need to load the model.

In [ ]:
model_name = "t5-base"


model = T5ForConditionalGeneration.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Then, we need to write the tokenizer function

In [ ]:
#The max_input_length and max_target_length are both settled before

def tokenize(batch):

    inputs = tokenizer(
        batch["input_text"],
        max_length=max_input_length,
        truncation=True
    )


    targets = tokenizer(
        batch["target_text"],
        max_length=max_target_length,
        truncation=True
    )


    inputs["labels"] = targets["input_ids"]

    return inputs

In [ ]:
train_dataset = train_dataset.map(
    tokenize,
    batched=True
)


val_dataset = val_dataset.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/3200 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Next, we need the Data Collator. During the training process, it combines multiple pieces of data into one batch and performs dynamic processing.

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

Next, set up TrainingArguments

In [ ]:
training_args = Seq2SeqTrainingArguments(

    output_dir="./t5_first6_results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=5e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    num_train_epochs=3,

    weight_decay=0.01,

    predict_with_generate=True,

    load_best_model_at_end=True,

    fp16=True,

    logging_steps=50,

    seed=42,
    data_seed=42
)

Then, create the trainer

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator
)

Then, apply the training dataset to the model

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.513602,1.337072
2,1.024467,1.324979
3,1.256358,1.330878


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=2400, training_loss=1.2201053714752197, metrics={'train_runtime': 1016.1916, 'train_samples_per_second': 9.447, 'train_steps_per_second': 2.362, 'total_flos': 5713889946992640.0, 'train_loss': 1.2201053714752197, 'epoch': 3.0})

Then, see the output for this

In [ ]:
#See the evaluate score
trainer.evaluate()

Training Loss,Validation Loss,Epoch
1.256358,1.324979,3


{'eval_loss': 1.3249785900115967}

Then, we would like to evaluate how this performs in the validation dataset. We use BLEU, ROUGE-L, and METEOR.

BLEU

In [ ]:
predictions = []

model.eval()

for i in tqdm(range(len(val_dataset))):

    inputs = {
        k: torch.tensor(v).unsqueeze(0).to(model.device)
        for k, v in val_dataset[i].items()
        if k in ["input_ids", "attention_mask"]
    }

    with torch.no_grad():

        generated_ids = model.generate(
            **inputs,
            max_length=64,
            num_beams=4
        )
    pred = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    )

    predictions.append(pred)

100%|██████████| 400/400 [07:35<00:00,  1.14s/it]


In [ ]:
references = [
    x if isinstance(x, list) else [x]
    for x in val["spoiler"]
]

bleu = evaluate.load("bleu")
bleu_score = bleu.compute(
    predictions=predictions,
    references=references
)
print("BLEU:", bleu_score["bleu"])

BLEU: 0.21725512146301207


ROUGE-L

In [ ]:
scorer = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

scores = []

for pred, ref in zip(predictions, references):
    score = scorer.score(ref[0], pred)
    scores.append(score["rougeL"].fmeasure)

rougeL = sum(scores) / len(scores)

print("ROUGE-L:", rougeL)

ROUGE-L: 0.3975723852671082


METEOR

In [ ]:
meteor = evaluate.load("meteor")

score = meteor.compute(
    predictions=predictions,
    references=references
)

print(score["meteor"])

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


0.4372861815085303


Take a manual inspection of the spoiler effect generated

In [ ]:
for i in range(5):

    sample = val.iloc[i]

    inputs = tokenizer(
        sample["input_text"],
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    ).to(model.device)


    generated_ids = model.generate(
        **inputs,
        max_length=64,
        num_beams=4
    )


    pred = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    )


    print("="*50)
    print("TRUE:")
    print(sample["target_text"])

    print("PRED:")
    print(pred)

TRUE:
some of the plot elements are so disturbing that they are making him feel sick
PRED:
too dark
TRUE:
"intentionally" could transform a court case against Phoenix-area Sheriff Joe Arpaio from civil charges to a criminal prosecution
PRED:
"intentionally"
TRUE:
20%
PRED:
between $5 and $20
TRUE:
Alan Rickman & Rupert Grint CBGB
PRED:
Alan Rickman as bemused owner Hilly Kristal
TRUE:
a man who swallowed a 64GB microSD card and then pooped it into a strainer
PRED:
he couldn't puke it back up, and therefore had to poop it into a pasta strainer and then plug it in to a computer to see if the client’s footage was intact


Then, we use the model on the test dataset

In [ ]:
test_inputs = tokenizer(
    test["input_text"].tolist(),
    max_length=1024,
    truncation=True,
    padding=True,
    return_tensors="pt"
)

test_inputs = {
    k: v.to(model.device)
    for k, v in test_inputs.items()
}

In [ ]:
#We can also try to change the hyperparameter to improve the prediction.
model.eval()

test_predictions = []

batch_size = 4

for i in tqdm(range(0, len(test), batch_size)):

    batch_texts = test["input_text"].iloc[i:i+batch_size].tolist()

    inputs = tokenizer(
        batch_texts,
        max_length=1024,
        truncation=True,
        padding=True,
        return_tensors="pt"
    ).to(model.device)


    with torch.no_grad():

        generated_ids = model.generate(
            **inputs,
            max_length=64,
            num_beams=4
        )


    decoded = [
        tokenizer.decode(
            ids,
            skip_special_tokens=True
        )
        for ids in generated_ids
    ]

    test_predictions.extend(decoded)


100%|██████████| 100/100 [03:30<00:00,  2.10s/it]


In [ ]:
len(test_predictions)

400

In [ ]:
for i in range(5):
    print("Prediction", i, ":")
    print(test_predictions[i])
    print()

Prediction 0 :
He has balloons and a sign in hand that reads, "Heard urine need of a kidney, want mine?"

Prediction 1 :
giving at the expense of your own well-being damages your chance of long-term success

Prediction 2 :
Have a Bunch of Money

Prediction 3 :
Braconid, meaning "any of numerous wasps of the family Braconidae, the larvae of whichare parasitic on aphids and on the larvae of moths, butterflies, beetles."

Prediction 4 :
Cured egg yolks



In [ ]:
#Then we have the submission

submission_task2 = pd.DataFrame({
    "id": test["id"],
    "spoiler": test_predictions
})

#Check the head of the submission
submission_task2.head()

,id,spoiler
0,0,"He has balloons and a sign in hand that reads,..."
1,1,giving at the expense of your own well-being d...
2,2,Have a Bunch of Money
3,3,"Braconid, meaning ""any of numerous wasps of th..."
4,4,Cured egg yolks


In [ ]:
submission_task2.to_csv(
    "prediction_task2.csv",
    index=False
)